# Case Study 8.2 - Sentence Pair Classification with BERT

In the second part of case study 8.2 we use our generated dataset to demonstrate a novel classification task that can be done effectively by fine-tuning BERT langauge models.

## 06 - Prepare the data

Set the data up ready for the classification task described in the chapter

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

In [ ]:
df = pd.read_csv("data/translated_expressions.csv")
fdf = df[df['has_idiom']==True].copy()
fdf['label'] = np.where(fdf['instruction']=="IdiomLiteral", 1, 0)

In [ ]:
repeats = fdf['translation'].value_counts().reset_index()
repeats = repeats[repeats['count']>1]

In [ ]:
values_to_drop = list(repeats['translation'])
fdf_clean = fdf[~fdf['translation'].isin(values_to_drop)]

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
split = gss.split(fdf_clean, groups=fdf_clean['ID'])
train_inds, test_inds = next(split)

In [ ]:
train = fdf_clean.iloc[train_inds]
test = fdf_clean.iloc[test_inds]

In [ ]:
train.to_csv("data/idiom_classifier_train.csv", index=False)
test.to_csv("data/idiom_classifier_test.csv", index=False)

## 07 - Build a Few Shot Classifier 

We will use a few-shot classifier as a baseline model for this task.

In [ ]:
import re
import json
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import BaseOutputParser
from langchain_core.output_parsers import JsonOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
model_name = "gemini-2.5-pro"
model = ChatGoogleGenerativeAI(model=model_name)

class Translation(BaseModel):
    literal_only: bool = Field(
        description="A Boolean value that indicates if the translation is\
                     an erroneous literal translation of the sentence. This\
                     value should be True only if the translation ignores an\
                     idiom in the sentence and translates it directly."
    )

parser = JsonOutputParser(pydantic_object=Translation)

In [ ]:
examples = """
English: Don't break the bank, but you can still paint the town red on a shoestring budget with a little creativity and planning!
Spanish: No rompas el banco, pero aún puedes pintar la ciudad de rojo con un presupuesto mínimo con un poco de creatividad y planificación.
literal_only: True
---
English: Don't break the bank, but you can still paint the town red on a shoestring budget with a little creativity and planning!
Spanish: No gastes de más, pero aún puedes pintar la ciudad de rojo con un presupuesto ajustado gracias a un poco de creatividad y planificación.
literal_only: False
"""

In [ ]:
prompt = PromptTemplate(
   template = """
     You are an expert on translations between English and Spanish.
     Your job is to determine whether the translation from English
     to Spanish below has an incorrect literal translation of text
     that uses an English idiom.
     Here are some examples:
     ---
     {few_shot_examples}
     ---
     English: {ENGLISH}
     Spanish: {SPANISH}
     ---
     {format_instructions}
     """,
   input_variables=[ "ENGLISH", "SPANISH"],
   partial_variables={
      "few_shot_examples": examples,
      "format_instructions":parser.get_format_instructions()
   }
)

In [ ]:
chain = prompt | model | parser

dataset = "data/idiom_classifier_test.csv"
df = pd.read_csv(dataset)

results = pd.DataFrame()

records = [x for i,x in df.iterrows()]

In [ ]:
for r in tqdm(records, desc=f"Processing Idiom Classifications for {dataset}"):

         record = {"ENGLISH": r['text'], "SPANISH": r['translation']}
         try:
             response = chain.invoke(record)
             r['literal_only'] = int(response['literal_only'])
         except Exception as e:
             print("Error processing record", e)
             r['literal_only'] = np.nan
         results = pd.concat([results, pd.DataFrame([r])], ignore_index=True)

results.to_csv("data/idiom_test_few_shot_model.csv")

### Evaluate the few-shot model

In [ ]:
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
import numpy as np

In [ ]:
y_true = list(results['label'])
y_pred = list(results['literal_only'])

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)

print(f"Precision: \t{round(precision,2)}")
print(f"Recall: \t{round(recall,2)}")

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Idiom Positive', 'Idiom Literal'])
disp.plot()
plt.savefig("results/few_shot_cm.png")

## 08 - Fine-Tune BERT

Finally we use our dataset to fine-tune a BERT model for the classification task.
This code was executed inside GCP on a (VM) instance with a pair of NVIDIA T4 GPUs.

In [ ]:
from datasets import ClassLabel, Value
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TextClassificationPipeline
from transformers import TrainingArguments, Trainer
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_curve
from sklearn.metrics import PrecisionRecallDisplay
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import evaluate
import torch
import sys

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("CUDA is available. Using GPU.")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

BUCKET_PATH = "gs://idiom_translations/"

In [ ]:
train = pd.read_csv(BUCKET_PATH + "idiom_classifier_train.csv")
test = pd.read_csv(BUCKET_PATH + "idiom_classifier_test.csv")

train_dataset = Dataset.from_pandas(train)
test_dataset = Dataset.from_pandas(test)

In [ ]:
hf_model = "google-bert/bert-base-multilingual-uncased"
max_length = 512

tokenizer = AutoTokenizer.from_pretrained(hf_model)

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"], examples["translation"],
        padding='max_length', truncation=True,
        max_length=max_length, return_tensors="pt"
    )

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

In [ ]:
roc_auc_score = evaluate.load("roc_auc")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    p = precision_metric.compute(predictions=predictions, references=labels)
    r = recall_metric.compute(predictions=predictions, references=labels)
    probs = np.exp(logits) / np.sum(np.exp(logits), axis=1, keepdims=True)
    auc = roc_auc_score.compute(prediction_scores=probs[:, 1], references=labels)
    return auc | p | r

In [ ]:
epochs = 10
dropout = 0.1
warmup=100

model = AutoModelForSequenceClassification.from_pretrained(
           hf_model,
           num_labels=2,
           hidden_dropout_prob=dropout,
           ignore_mismatched_sizes=True
)

In [ ]:
dirname = f"./checkpoints/BERT_{dropout}_{warmup}_{epochs}"
training_args = TrainingArguments(
      output_dir=dirname,
      num_train_epochs=epochs,                # total number of training epochs
      warmup_steps=warmup,                       # number of warmup steps for learning rate scheduler
      weight_decay=0.01,                      # strength of weight decay
      logging_dir='./logs',                   # directory for storing logs
      logging_steps=100,
      eval_strategy='epoch'
)

In [ ]:
trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=tokenized_train_dataset,
      eval_dataset=tokenized_test_dataset,
      compute_metrics=compute_metrics,
)
trainer.train()
results = trainer.evaluate()
print(results)

In [ ]:
# Create a DataLoader
batch_size = 16
tokenized_test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'token_type_ids', 'labels'])
dataloader = DataLoader(tokenized_test_dataset, batch_size=batch_size, shuffle=False)
model.to(device)
model.eval()

all_preds = []
all_labels = []
all_scores = []
with torch.no_grad(): # Disable gradient calculation during evaluation
      for batch in dataloader:
         input_ids = batch['input_ids'].to(device)
         attention_mask = batch['attention_mask'].to(device)
         token_type_ids = batch['token_type_ids'].to(device)
         labels = batch['labels'].to(device)
         outputs = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
         logits = outputs.logits.cpu().numpy()
         predictions = np.argmax(logits, axis=-1)
         probs = np.exp(logits) / np.sum(np.exp(logits), axis=1, keepdims=True)
         probabilities = probs[:,1]
         all_preds.extend(predictions)
         all_labels.extend(labels.cpu().numpy())
         all_scores.extend(probabilities)

In [ ]:
# Calculate metrics
auc_score = roc_auc_score.compute(prediction_scores=all_scores, references=all_labels)['roc_auc']
p = precision_metric.compute(predictions=all_preds, references=all_labels)['precision']
r = recall_metric.compute(predictions=all_preds, references=all_labels)['recall']
accuracy = accuracy_score(all_labels, all_preds)
fpr, tpr, thresholds = roc_curve(all_labels, all_scores)

In [ ]:
posies = { "positives" : np.sum(all_labels)}
records = {"records" : len(all_labels)}
pospred = {"preds" : np.sum(all_preds) }
print(f"Test AUC: {round(auc_score, 3)}")
print(f"Precision: {round(p, 3)}")
print(f"Recall: {round(r, 3)}")
print(f"Test Data has {posies} positive records of {records} - {pospred} predicted positive")

In [ ]:
with open("BERT_Fine_Tune_Results.txt", "a") as f:
      f.write(f"Model : {hf_model}\n")
      f.write(f"Epochs: {epochs}\n")
      f.write(f"Dropout: {dropout}\n")
      f.write(f"Warmup: {warmup}\n")
      f.write(f"AUC: {round(auc_score, 3)}\n")
      f.write(f"Accuracy: {round(accuracy,3)}\n")
      f.write(f"Precision: {round(p, 3)}\n")
      f.write(f"Recall: {round(r, 3)}\n")
      f.write(f"---------------\n")

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % auc_score)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.savefig("results/idiom_transalation_ROC.png")

In [ ]:
plt.close()

display = PrecisionRecallDisplay.from_predictions(
    all_labels, all_scores, name="BERT", plot_chance_level=True
)
_ = display.ax_.set_title("Idiom Classifier Precision-Recall Curve")
plt.savefig("results/idiom_transalation_PRC.png")